## Scenario 1: A single data scientist participating in an ML competition

MLflow setup:
* Tracking server: no
* Backend store: local filesystem
* Artifacts store: local filesystem

The experiments can be explored locally by launching the MLflow UI.

We are not running any mlflow server or ui just importing mlflow and it automatically set a tracking uri 
within the same folder, later I will delete it but the output of file will show us information.

In [1]:
import mlflow

In [2]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'file:///home/mshifa/workspace/zoomcamp/repo_clone/mlops-zoomcamp2025/02-experiment-tracking/running-mlflow-examples/mlruns'


In [4]:
mlflow.search_experiments() # we have always a defaults experiment inside the mlruns

[<Experiment: artifact_location='file:///home/mshifa/workspace/zoomcamp/repo_clone/mlops-zoomcamp2025/02-experiment-tracking/running-mlflow-examples/mlruns/0', creation_time=1751385653158, experiment_id='0', last_update_time=1751385653158, lifecycle_stage='active', name='Default', tags={}>]

### Creating an experiment and logging a new run

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("my-experiment-1")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, artifact_path="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2025/07/01 21:04:12 INFO mlflow.tracking.fluent: Experiment with name 'my-experiment-1' does not exist. Creating a new experiment.
2025/07/01 21:04:18 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


default artifacts URI: 'file:///home/mshifa/workspace/zoomcamp/repo_clone/mlops-zoomcamp2025/02-experiment-tracking/running-mlflow-examples/mlruns/900216742306171145/14fc20fc107248a3ba5fe7ea40d07315/artifacts'


In [6]:
mlflow.search_experiments() # after creating experiment let's search

[<Experiment: artifact_location='file:///home/mshifa/workspace/zoomcamp/repo_clone/mlops-zoomcamp2025/02-experiment-tracking/running-mlflow-examples/mlruns/900216742306171145', creation_time=1751385852831, experiment_id='900216742306171145', last_update_time=1751385852831, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='file:///home/mshifa/workspace/zoomcamp/repo_clone/mlops-zoomcamp2025/02-experiment-tracking/running-mlflow-examples/mlruns/0', creation_time=1751385653158, experiment_id='0', last_update_time=1751385653158, lifecycle_stage='active', name='Default', tags={}>]

### Interacting with the model registry

In [7]:
from mlflow.tracking import MlflowClient


client = MlflowClient()

In [9]:
from mlflow.exceptions import MlflowException

try:
    print(client.search_registered_models())
except MlflowException:
    print("It's not possible to access the model registry :(")

[]


## Interacting with the Model Registry: Previously Not Possible, Now Supported

In the lecture, it was mentioned that the MLflow Model Registry can be used in such scenarios.  
Let me try this out myself to see whether it works with a local MLflow setup.

**MLflow setup:**
- Tracking server: ❌ None (local mode)
- Backend store: Local filesystem
- Artifact store: Local filesystem

In [10]:
# let's search experiment
client.search_experiments()

[<Experiment: artifact_location='file:///home/mshifa/workspace/zoomcamp/repo_clone/mlops-zoomcamp2025/02-experiment-tracking/running-mlflow-examples/mlruns/900216742306171145', creation_time=1751385852831, experiment_id='900216742306171145', last_update_time=1751385852831, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='file:///home/mshifa/workspace/zoomcamp/repo_clone/mlops-zoomcamp2025/02-experiment-tracking/running-mlflow-examples/mlruns/0', creation_time=1751385653158, experiment_id='0', last_update_time=1751385653158, lifecycle_stage='active', name='Default', tags={}>]

In [11]:
client.get_experiment_by_name(name="my-experiment-1")

<Experiment: artifact_location='file:///home/mshifa/workspace/zoomcamp/repo_clone/mlops-zoomcamp2025/02-experiment-tracking/running-mlflow-examples/mlruns/900216742306171145', creation_time=1751385852831, experiment_id='900216742306171145', last_update_time=1751385852831, lifecycle_stage='active', name='my-experiment-1', tags={}>

In [12]:
# Check if the experiment has run
runs_all = client.search_runs("900216742306171145")
print(f"Total runs found: {len(runs_all)}")

Total runs found: 1


In [13]:
runs = client.search_runs(
    experiment_ids="900216742306171145",
    max_results=5
)

for run in runs:
    print(f"run id: {run.info.run_id}, metrics: {run.data.metrics}")

run id: 14fc20fc107248a3ba5fe7ea40d07315, metrics: {'accuracy': 0.96}


In [14]:
import mlflow
MLFLOW_TRACKING_URI = mlflow.get_tracking_uri()
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [16]:
run_id = "14fc20fc107248a3ba5fe7ea40d07315"
model_uri = f"runs:/{run_id}/model"
mlflow.register_model(model_uri=model_uri, name="iris-classifier")

Successfully registered model 'iris-classifier'.
Created version '1' of model 'iris-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1751386642564, current_stage='None', description=None, last_updated_timestamp=1751386642564, name='iris-classifier', run_id='14fc20fc107248a3ba5fe7ea40d07315', run_link=None, source='file:///home/mshifa/workspace/zoomcamp/repo_clone/mlops-zoomcamp2025/02-experiment-tracking/running-mlflow-examples/mlruns/900216742306171145/14fc20fc107248a3ba5fe7ea40d07315/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=1>

In [17]:
model_name = "iris-classifier"
latest_versions = client.get_latest_versions(name=model_name)

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 1, stage: None


/tmp/ipykernel_282527/2355489280.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [18]:
model_version = 1
new_stage = "Staging"
client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

/tmp/ipykernel_282527/1600074043.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1751386642564, current_stage='Staging', description=None, last_updated_timestamp=1751391604658, name='iris-classifier', run_id='14fc20fc107248a3ba5fe7ea40d07315', run_link=None, source='file:///home/mshifa/workspace/zoomcamp/repo_clone/mlops-zoomcamp2025/02-experiment-tracking/running-mlflow-examples/mlruns/900216742306171145/14fc20fc107248a3ba5fe7ea40d07315/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=1>

In [19]:
from datetime import datetime

date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_stage} on {date}"
)

<ModelVersion: aliases=[], creation_timestamp=1751386642564, current_stage='Staging', description='The model version 1 was transitioned to Staging on 2025-07-01', last_updated_timestamp=1751391625646, name='iris-classifier', run_id='14fc20fc107248a3ba5fe7ea40d07315', run_link=None, source='file:///home/mshifa/workspace/zoomcamp/repo_clone/mlops-zoomcamp2025/02-experiment-tracking/running-mlflow-examples/mlruns/900216742306171145/14fc20fc107248a3ba5fe7ea40d07315/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=1>

## That's it we can easily managed the model registry even with mlflow locally